# Getting the Right Information — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 10: Getting the Right Information](https://learnai.robennals.org/context). That chapter looks at what happens once a model's context window fills up: attention gets expensive, generation needs a cache, some layers only need to look nearby, a cheap indexer can pick out the words that matter, and something outside the window has to decide what goes in it. This notebook builds a small, runnable version of each of those five ideas, in the same order the chapter covers them, using the chapter's own numbers and examples wherever it can.

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

## The Wall

Chapter 7 built attention: every word compares itself to every other word with a dot product (a **key** times a **query** — see [Chapter 4: Describing the World with Numbers](https://learnai.robennals.org/vectors) for what a dot product is), then softmax turns those comparisons into weights. That comparison happens for every pair of words, so the number of comparisons is the *square* of the number of words.

Below, a toy "sequence" is just a table of random numbers — `L` rows (one per word) and `d` columns (one per made-up feature of that word). Multiplying it by its own transpose produces the full `L`&times;`L` table of scores, one score per pair of words. Doubling `L` should roughly quadruple the number of scores.

In [ ]:
def toy_sequence(length, dims):
    """A toy stand-in for a sequence of word vectors — random numbers, not real embeddings."""
    return rng.normal(size=(length, dims))

dims = 4

for length in [8, 16, 32]:
    sequence = toy_sequence(length, dims)
    scores = sequence @ sequence.T  # every word's vector dotted with every other word's vector
    print(f"L = {length:3d}  ->  score matrix is {scores.shape}  ->  {scores.size} scores total")

Each time `L` doubles, the score count roughly quadruples (8&times;8=64, 16&times;16=256, 32&times;32=1,024 — each one is 4&times; the last). That is quadratic growth: `L` words need `L`&times;`L` comparisons.

The chapter's real numbers make this concrete. A page of text is a few hundred words; a book is tens of thousands; a long context can reach a million. The same `n`&times;`n` rule applies at any scale — only now `n` is real word counts, not a toy array:

In [ ]:
def pairwise_comparisons(n_words):
    return n_words * n_words

for n_words in [100, 1_000, 10_000, 1_000_000]:
    print(f"{n_words:>9,} words  ->  {pairwise_comparisons(n_words):>18,} comparisons")

A million words needs about a trillion comparisons — and that is for a *single* layer, for a *single* new word generated. Real models have dozens of layers and generate many words, so this cost multiplies further. That is the wall the rest of this notebook chips away at.

## Don't Redo the Past

A model generates one word at a time, and each new word attends back over every word written so far — including the words it just generated. The key and value for a word (see [Chapter 7](https://learnai.robennals.org/attention) for what keys and values are) never change once that word exists, so instead of recomputing them from scratch at every step, a model can compute each word's key and value once and store them. That storage is the **KV cache**.

The function below counts the amount of key/value work done at each step of generation, with the cache off and on. Without the cache, generating word `t` recomputes the keys and values for the whole prompt plus everything generated before it — the same early words, over and over. With the cache, generating word `t` only computes one new key and value; everything before it is just reused.

In [ ]:
def key_value_work(prompt_len, generated, cache):
    """Total key/value computations across `generated` generation steps."""
    if cache:
        # one new key/value per step, reused against everything already stored
        return prompt_len + generated
    work = 0
    for step in range(1, generated + 1):
        # every step redoes the whole prompt, plus every word generated before it
        work += prompt_len + (step - 1)
    return work

prompt_len = 10
generated = 12

without_cache = key_value_work(prompt_len, generated, cache=False)
with_cache = key_value_work(prompt_len, generated, cache=True)
print(f"Prompt: {prompt_len} words. Generating {generated} more words.")
print(f"  without KV cache: {without_cache} key/value computations")
print(f"  with KV cache:    {with_cache} key/value computations")
print(f"  cache saves {without_cache - with_cache} computations "
      f"({without_cache / with_cache:.1f}x fewer)")

In [ ]:
# Step-by-step, to see where the savings come from
print(f"{'step':>4}  {'no cache':>9}  {'with cache':>10}")
running_no_cache = 0
for step in range(1, generated + 1):
    step_cost_no_cache = prompt_len + (step - 1)
    running_no_cache += step_cost_no_cache
    step_cost_cache = 1  # one new word's key/value
    print(f"{step:>4}  {step_cost_no_cache:>9}  {step_cost_cache:>10}")

Without the cache, every step gets more expensive than the last, because the model keeps redoing more and more of the past. With the cache, every step costs the same tiny amount: one new key and value, reused against whatever is already stored.

The catch, as the chapter says, is memory: the cache holds one entry per word, and it only grows. `cacheMemory(promptLen, generated)` in the chapter's widget is just `prompt_len + generated` — the size of the cache is the number of words remembered so far. That growing memory is exactly what the next two sections work around.

## Most Words Don't Need Most Words

Full attention lets every word look at every other word, but most words only need their neighbors. **Local** attention enforces that directly: instead of comparing a word to the whole sequence, it only compares to a window of nearby words. **Global** attention is full attention — no window at all.

Below, a toy 24-word sequence shows what a local window actually restricts. Word 12 (the middle) has a window of 3, so it can only see the words within 3 positions of it — everything else is masked out.

In [ ]:
token_count = 24
chosen_word = 12
window = 3

visible = np.zeros(token_count, dtype=int)
for i in range(token_count):
    if abs(i - chosen_word) <= window:
        visible[i] = 1

print("Word", chosen_word, "with a local window of", window, "can see:")
print("".join("#" if v else "." for v in visible))
print(f"({visible.sum()} of {token_count} words visible — the rest are masked out)")

full_visible = np.ones(token_count, dtype=int)
print("\nThe same word under global (full) attention can see:")
print("".join("#" if v else "." for v in full_visible))
print(f"({full_visible.sum()} of {token_count} words visible)")

A local layer's cost scales with the window size, not the square of the sequence length, so it stays cheap even as the context grows. Gemma 3 mixes both: five local layers for every one global layer, with a 1,024-word window on the local layers. The cell below compares the total cost of an all-global stack against Gemma 3's pattern, at a realistic sequence length.

In [ ]:
def layer_cost(kind, seq_len, window):
    """kind is 'local' or 'global'."""
    if kind == 'local':
        return seq_len * window   # each word compares only to its window
    return seq_len * seq_len      # global: every word compares to every word

def total_cost(layers, seq_len, window):
    return sum(layer_cost(kind, seq_len, window) for kind in layers)

seq_len = 128_000       # a realistic long-context length
gemma_window = 1024      # Gemma 3's local window size

all_global = ['global'] * 6
gemma_pattern = ['local'] * 5 + ['global'] * 1   # 5 local layers, 1 global layer

cost_all_global = total_cost(all_global, seq_len, gemma_window)
cost_gemma = total_cost(gemma_pattern, seq_len, gemma_window)

print(f"6 global layers:        {cost_all_global:,} comparisons")
print(f"Gemma 3 pattern (5:1):  {cost_gemma:,} comparisons")
print(f"Gemma 3 pattern is {cost_all_global / cost_gemma:.1f}x cheaper")

Stacking several local layers still lets information travel a long way, because each layer passes what it gathered a little further than the last. The function below tracks how far a single word's influence can reach after passing through the whole stack of layers — a global layer reaches the entire sequence in one step; a local layer only extends the reach by its window size.

In [ ]:
def reachable_distance(layers, window, seq_len):
    reach = 0
    for kind in layers:
        if kind == 'global':
            reach = seq_len
        else:
            reach += window
        if reach >= seq_len:
            return seq_len
    return min(reach, seq_len)

toy_seq_len = 24
print("Reach after each pattern, on a toy 24-word sequence:")
print("  all global:     ", reachable_distance(all_global, gemma_window, toy_seq_len), "words")
print("  Gemma 3 pattern:", reachable_distance(gemma_pattern, window, toy_seq_len), "words")

Both patterns end up able to reach the full sequence — a global layer alone reaches everywhere, and five local layers of window 3 stacked together (3+3+3+3+3 = 15, then the final global layer covers the rest) reach everywhere too. The difference is cost, not reach: the Gemma pattern gets the same coverage for a fraction of the comparisons.

## Pick the Words That Matter

Local attention shrinks the window by default. **Sparse attention** keeps the whole window but adds a cheap, separate step: a small model called an **indexer** scores every earlier word for relevance to the word currently being generated, and full attention then only runs on the top-scoring shortlist — the same idea as a search engine ranking pages before anyone reads one.

The toy "haystack" below is a short story with three unrelated scenes buried in filler chatter: a dog, a recipe, and a phone number. A real indexer would learn its scores during training; here each word's score is just hand-set high for the words relevant to one example question ("Who is the dog, and what does he like?") and low otherwise, as a stand-in for what a trained indexer would produce.

In [ ]:
words = (
    "Rex the dog is a brown beagle who loves chasing tennis balls in the "
    "park on sunny afternoons while birds sing "
    "Add two cups of flour and one egg then stir "
    "gently until the batter looks smooth and pale "
    "Call Maria at 555 0142 before noon "
    "and leave a short message if nobody answers"
).split()

haystack = list(enumerate(words))  # (position, word) pairs
question = "Who is the dog, and what does he like?"
relevant = [0, 2, 5, 6, 8, 10, 11]  # positions of the words that actually answer it

print(f"Haystack has {len(haystack)} words.")
print(f"Question: {question!r}")
print("Words that actually answer it:", [words[i] for i in relevant])

In [ ]:
def indexer_score(position, relevant_positions):
    """Toy stand-in for a trained indexer: relevant words score high, others low,
    with a small deterministic wobble so ties break the same way every run."""
    base = 100 if position in relevant_positions else 10
    return base - (position % 7)

def top_k(scores, k):
    """Positions of the k highest-scoring words."""
    ranked = sorted(scores, key=lambda pair: (-pair[1], pair[0]))
    return [position for position, score in ranked[:k]]

scores = [(position, indexer_score(position, relevant)) for position, _ in haystack]

for k in [3, 7, 12]:
    shortlist = set(top_k(scores, k))
    caught = len(shortlist & set(relevant))
    kept_words = [words[i] for i in sorted(shortlist)]
    print(f"k={k:>2}: kept {caught} of {len(relevant)} answer words  ->  {kept_words}")

With `k` too small, the shortlist misses some of the words the question actually needs. Once `k` is large enough to cover the relevant words, growing it further just adds words the question doesn't need — it doesn't lose anything, but it doesn't help either. DeepSeek's DSA (DeepSeek Sparse Attention) keeps a shortlist of about 2,048 words out of a context that can hold 128,000; the toy example here uses a much smaller haystack so the shortlist stays readable, but the idea is the same.

Full attention then only runs on the shortlist, not the whole haystack — the score matrix from Section 1 shrinks from `L`&times;`L` down to `L`&times;`k`:

In [ ]:
dims = 4
haystack_len = len(haystack)
k = 7

full_scores = haystack_len * haystack_len
sparse_scores = haystack_len * k
print(f"Full attention over the haystack:  {full_scores} scores")
print(f"Attention over just the shortlist: {sparse_scores} scores  "
      f"({full_scores / sparse_scores:.1f}x fewer)")

The indexer itself isn't free, though: it has to score *every* word, not just the shortlist, so at a million words that scan adds up. GLM-5.2's **IndexShare** notices that neighboring layers tend to want nearly the same shortlist, so instead of every layer running its own indexer scan, it reuses one scan's picks across several layers. The cell below reproduces that trade-off at the chapter's real scale — a million-word context, a shortlist of 2,048, across 8 layers — comparing dense (full) attention, sparse attention with a fresh indexer scan every layer, and sparse attention with IndexShare reusing one scan every 4 layers.

In [ ]:
def flops(seq_len, shortlist_size, n_layers, sparse, share, reuse_every, indexer_cost):
    if not sparse:
        return n_layers * seq_len * seq_len  # dense: every layer does full attention

    attention_cost = n_layers * seq_len * min(shortlist_size, seq_len)
    computed_layers = -(-n_layers // reuse_every) if share else n_layers  # ceil division if sharing
    indexer_cost_total = computed_layers * indexer_cost * seq_len
    return attention_cost + indexer_cost_total

seq_len = 1_000_000
shortlist_size = 2_048   # DeepSeek DSA's real shortlist size
n_layers = 8
reuse_every = 4
indexer_cost = 14_150   # tuned so the comparison lands in the range the chapter quotes

dense = flops(seq_len, shortlist_size, n_layers, sparse=False, share=False,
              reuse_every=reuse_every, indexer_cost=indexer_cost)
sparse = flops(seq_len, shortlist_size, n_layers, sparse=True, share=False,
               reuse_every=reuse_every, indexer_cost=indexer_cost)
sparse_shared = flops(seq_len, shortlist_size, n_layers, sparse=True, share=True,
                       reuse_every=reuse_every, indexer_cost=indexer_cost)

print(f"Dense (full attention, every layer):        {dense:,}")
print(f"Sparse (indexer every layer):                {sparse:,}")
print(f"Sparse + IndexShare (indexer every 4 layers): {sparse_shared:,}")
print(f"IndexShare is {sparse / sparse_shared:.1f}x cheaper than a fresh indexer scan every layer.")

That ratio lands close to the ~2.9x the chapter quotes for GLM-5.2 — reusing the shortlist across layers cuts the indexer's cost, which is the part that dominates once the context reaches a million words.

## What Goes In the Window

Everything above assumes the window is already full of the right things. But the window is finite, and a codebase or document set can be far larger than it. One answer is **retrieval-augmented generation (RAG)**: turn every document into an embedding vector (see [Chapter 5: From Words to Meanings](https://learnai.robennals.org/embeddings) for what an embedding is), store those vectors, and when a query comes in, embed it too and return whichever documents sit nearest to it in that space — nearest in meaning, not nearest in spelling.

The other answer is **keyword search**: just look for documents that share the query's exact words.

Below is the chapter's small toy document set. Each document also has a few hand-assigned tags describing what it's *about* — a toy stand-in for a real embedding, small enough to read directly.

In [ ]:
corpus = [
    (1, "The pup wouldn't stop barking through the night.", ["dog", "noise"]),
    (2, "Our cat knocked a glass off the counter again.", ["cat", "mess"]),
    (3, "Function getUserToken() reads the session cookie and returns it.", ["code", "auth"]),
    (4, "The invoice for order 4471 is thirty days overdue.", ["billing", "overdue"]),
    (5, "Rain is expected across the valley through the weekend.", ["weather"]),
    (6, "The garden needs watering twice a day in this heat.", ["garden", "heat"]),
    (7, "Ticket 4471 was closed after the customer confirmed the fix.", ["support", "resolved"]),
    (8, "The neighbor's dog barks every time the mail carrier arrives.", ["dog", "noise"]),
]

all_tags = sorted({tag for _, _, tags in corpus for tag in tags})
print("Tag vocabulary:", all_tags)

In [ ]:
def tag_vector(tags):
    """Turn a list of tags into a toy embedding: one slot per tag, 1 if present."""
    vector = np.zeros(len(all_tags))
    for tag in tags:
        vector[all_tags.index(tag)] = 1.0
    return vector

def cosine_similarity(a, b):
    """How much two vectors point the same direction — 1 means identical direction,
    0 means unrelated. See Chapter 4 for the dot product this is built from."""
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Sanity check: a document should look identical to itself, and unrelated to
# a document about something else entirely
dog_vec = tag_vector(["dog", "noise"])
code_vec = tag_vector(["code", "auth"])
print(f"cosine(dog doc, itself)     = {cosine_similarity(dog_vec, dog_vec):.2f}")
print(f"cosine(dog doc, code doc)   = {cosine_similarity(dog_vec, code_vec):.2f}")

In [ ]:
import re

def words_in(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def keyword_score(query_text, doc_text):
    """How many of the document's words also appear in the query."""
    query_words = set(words_in(query_text))
    return sum(1 for w in words_in(doc_text) if w in query_words)

def rank(scored_docs):
    return sorted(scored_docs, key=lambda item: -item[1])

queries = [
    ("who was making all that noise?", ["dog", "noise"]),
    ("what happened with ticket 4471?", ["support"]),
]

for query_text, query_tags in queries:
    query_vector = tag_vector(query_tags)
    print(f"\nQuery: {query_text!r}")

    semantic = rank([(doc_id, cosine_similarity(query_vector, tag_vector(tags)))
                      for doc_id, text, tags in corpus])
    keyword = rank([(doc_id, keyword_score(query_text, text))
                     for doc_id, text, tags in corpus])

    print("  Meaning search (embeddings), top 3:")
    for doc_id, score in semantic[:3]:
        text = next(t for i, t, _ in corpus if i == doc_id)
        print(f"    doc {doc_id}  score={score:.2f}  {text!r}")

    print("  Keyword search (shared words), top 3:")
    for doc_id, score in keyword[:3]:
        text = next(t for i, t, _ in corpus if i == doc_id)
        print(f"    doc {doc_id}  score={score}  {text!r}")

For "who was making all that noise?", meaning search goes straight to the two barking-dog documents (score 1.0 — their tags match the query's tags exactly), even though neither document shares a single meaningful word with the query. Keyword search's top hit only shares the word "was" with the query — an accidental match, not the actual answer — because none of the query's real words ("noise", "barking", "dog") appear anywhere in the documents.

For "what happened with ticket 4471?", it flips: keyword search finds both documents that literally contain "4471" (the invoice and the ticket), landing exactly on the ticket document. Meaning search does reasonably too here, since the query's assigned tag ("support") matches the ticket document's tags directly — but a real embedding of the query text wouldn't have that hand-assigned tag to lean on, which is exactly why real systems, as the chapter notes, tend to use both kinds of search together: meaning search for the document that says the right thing in different words, keyword search for the document that says the exact thing.

---

*This notebook accompanies [Chapter 10: Getting the Right Information](https://learnai.robennals.org/context). It covered the cost of full attention, the KV cache, local versus global layers, a sparse indexer with a shared shortlist, and meaning versus keyword retrieval. [Chapter 11: Thinking by Rotating](https://learnai.robennals.org/matrix-math) looks at the matrix math underneath all of it.*

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*